In [1]:
import os
import zipfile
import gdown
from pathlib import Path

def download_and_extract():
    # Google Drive File ID from the link:
    # https://drive.google.com/file/d/1oMDFwlGwObPg0kmyFVGVoYEMtc8ubOWT/view
    file_id = "1oMDFwlGwObPg0kmyFVGVoYEMtc8ubOWT"
    url = f"https://drive.google.com/uc?id={file_id}"

    # Target directories
    raw_dir = Path("data/raw")
    raw_dir.mkdir(parents=True, exist_ok=True)

    zip_path = raw_dir / "rdd2022_india.zip"

    # Download
    print(f"Downloading dataset from Google Drive (ID: {file_id})...")
    try:
        gdown.download(url, str(zip_path), quiet=False)
    except Exception as e:
        print(f"Error downloading via gdown: {e}")
        print("Please check if gdown is installed and the Google Drive link is accessible.")
        return

    # Extract
    if zip_path.exists():
        print("Extracting dataset...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(raw_dir)
            print("Extraction completed successfully!")

            # Clean up the zip file after extraction
            os.remove(zip_path)
            print("Cleaned up the zip file.")
        except Exception as e:
            print(f"Error during extraction: {e}")
    else:
        print("Download failed, zip file not found.")

if __name__ == "__main__":
    download_and_extract()


Downloading...
From (original): https://drive.google.com/uc?id=1oMDFwlGwObPg0kmyFVGVoYEMtc8ubOWT
From (redirected): https://drive.google.com/uc?id=1oMDFwlGwObPg0kmyFVGVoYEMtc8ubOWT&confirm=t&uuid=5d3f01ba-1968-41c5-b400-a6999f06211c
To: /content/data/raw/rdd2022_india.zip
100%|██████████| 527M/527M [00:04<00:00, 119MB/s]


Extracting dataset...
Extraction completed successfully!
Cleaned up the zip file.


In [2]:
import os
import xml.etree.ElementTree as ET
import cv2
import random
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

CLASSES = ["D00", "D20", "D40"]

def get_iou(boxA, boxB):
    # box format: [xmin, ymin, xmax, ymax]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    if boxAArea + boxBArea - interArea == 0:
        return 0

    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

def check_overlap_with_any(candidate_box, ground_truth_boxes):
    for gt_box in ground_truth_boxes:
        if get_iou(candidate_box, gt_box) > 0.0:
            return True
    return False

def parse_xml(xml_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception as e:
        print(f"Error parsing XML {xml_path}: {e}")
        return None, []

    filename = root.find("filename").text

    # Get image size
    size_node = root.find("size")
    width = int(size_node.find("width").text)
    height = int(size_node.find("height").text)

    objects = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name in CLASSES:
            bndbox = obj.find("bndbox")
            xmin = int(float(bndbox.find("xmin").text))
            ymin = int(float(bndbox.find("ymin").text))
            xmax = int(float(bndbox.find("xmax").text))
            ymax = int(float(bndbox.find("ymax").text))

            # Make sure coordinates are within image boundaries
            xmin = max(0, xmin)
            ymin = max(0, ymin)
            xmax = min(width, xmax)
            ymax = min(height, ymax)

            if xmax > xmin and ymax > ymin:
                objects.append({
                    "class": name,
                    "box": [xmin, ymin, xmax, ymax]
                })

    return filename, objects

def process_dataset():
    raw_dir = Path("data/raw")
    processed_dir = Path("data/processed")

    # Locate images and annotations
    # RDD2022 India zip usually contains India/images/ and India/annotations/xmls/
    india_dir = None
    for p in raw_dir.glob("**/annotations/xmls"):
        india_dir = p.parent.parent
        break

    if not india_dir:
        print("Could not find India annotations directory. Please check extraction path.")
        return

    images_dir = india_dir / "images"
    annotations_dir = india_dir / "annotations" / "xmls"

    print(f"Found dataset at: {india_dir}")
    xml_files = list(annotations_dir.glob("*.xml"))
    print(f"Total xml annotation files: {len(xml_files)}")

    # Split xml files at image level to avoid data leakage
    train_xml, test_xml = train_test_split(xml_files, test_size=0.3, random_state=42)
    val_xml, test_xml = train_test_split(test_xml, test_size=0.5, random_state=42)

    splits = {
        "train": train_xml,
        "val": val_xml,
        "test": test_xml
    }

    # Initialize output class directories
    for split in splits:
        for cls in CLASSES + ["Normal"]:
            (processed_dir / split / cls).mkdir(parents=True, exist_ok=True)

    # Process each split
    for split_name, files in splits.items():
        print(f"Processing {split_name} split ({len(files)} images)...")

        for xml_path in tqdm(files):
            img_filename, objects = parse_xml(xml_path)
            if img_filename is None:
                continue

            img_path = images_dir / img_filename
            # Fallback in case of extensions mismatches
            if not img_path.exists():
                img_path = images_dir / (xml_path.stem + ".jpg")

            if not img_path.exists():
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                continue

            h, w, _ = img.shape
            gt_boxes = [obj["box"] for obj in objects]

            # 1. Crop actual damage patches
            for idx, obj in enumerate(objects):
                cls_name = obj["class"]
                box = obj["box"]
                xmin, ymin, xmax, ymax = box

                # Crop and save patch
                patch = img[ymin:ymax, xmin:xmax]
                if patch.size > 0:
                    patch_name = f"{xml_path.stem}_damage_{idx}_{cls_name}.jpg"
                    out_path = processed_dir / split_name / cls_name / patch_name
                    cv2.imwrite(str(out_path), patch)

            # 2. Sample random Normal patches
            # Try to crop a few non-overlapping background patches of average damage size (e.g. 128x128)
            patch_size = 128
            sampled_normal = 0
            # Limit the number of normal patches per image to keep classes balanced
            max_normal_per_image = max(1, len(objects))

            attempts = 0
            while sampled_normal < max_normal_per_image and attempts < 20:
                attempts += 1
                if w - patch_size <= 0 or h - patch_size <= 0:
                    break
                nx = random.randint(0, w - patch_size)
                ny = random.randint(0, h - patch_size)
                candidate_box = [nx, ny, nx + patch_size, ny + patch_size]

                if not check_overlap_with_any(candidate_box, gt_boxes):
                    patch = img[ny:ny+patch_size, nx:nx+patch_size]
                    if patch.size > 0:
                        patch_name = f"{xml_path.stem}_normal_{sampled_normal}.jpg"
                        out_path = processed_dir / split_name / "Normal" / patch_name
                        cv2.imwrite(str(out_path), patch)
                        sampled_normal += 1

    print("Patch preprocessing completed!")
    # Show stats
    for split in splits:
        print(f"\nStats for {split}:")
        for cls in CLASSES + ["Normal"]:
            path = processed_dir / split / cls
            count = len(list(path.glob("*.jpg")))
            print(f"  - {cls}: {count} patches")

if __name__ == "__main__":
    process_dataset()


Found dataset at: data/raw/India/train
Total xml annotation files: 7706
Processing train split (5394 images)...


100%|██████████| 5394/5394 [00:17<00:00, 301.08it/s]


Processing val split (1156 images)...


100%|██████████| 1156/1156 [00:03<00:00, 344.01it/s]


Processing test split (1156 images)...


100%|██████████| 1156/1156 [00:03<00:00, 335.16it/s]


Patch preprocessing completed!

Stats for train:
  - D00: 1110 patches
  - D20: 1434 patches
  - D40: 2219 patches
  - Normal: 7889 patches

Stats for val:
  - D00: 197 patches
  - D20: 280 patches
  - D40: 476 patches
  - Normal: 1641 patches

Stats for test:
  - D00: 248 patches
  - D20: 307 patches
  - D40: 492 patches
  - Normal: 1708 patches


In [3]:
!pwd
!ls -F

/content
data/  sample_data/


In [4]:
!python src/preprocessing/transforms.py

python3: can't open file '/content/src/preprocessing/transforms.py': [Errno 2] No such file or directory


In [5]:
!python src/models/baseline_cnn.py

python3: can't open file '/content/src/models/baseline_cnn.py': [Errno 2] No such file or directory


In [6]:
!python src/training/dataset.py

python3: can't open file '/content/src/training/dataset.py': [Errno 2] No such file or directory


In [7]:
!python -m src.training.train --config configs/baseline.yaml

/usr/bin/python3: Error while finding module specification for 'src.training.train' (ModuleNotFoundError: No module named 'src')


In [8]:
!ls outputs/baseline

ls: cannot access 'outputs/baseline': No such file or directory


In [9]:
!python -m src.evaluation.evaluate --config configs/baseline.yaml --model_path outputs/baseline/best_model.pth

/usr/bin/python3: Error while finding module specification for 'src.evaluation.evaluate' (ModuleNotFoundError: No module named 'src')
